# Day 7: CNN Training Pipeline Practice

Dataset → DataLoader → Model → Loss → Optimizer → Train → Validation

In [44]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [34]:
transform = transforms.ToTensor()

train_datasets = datasets.FashionMNIST('./data', train=True, download=False, transform=transform)
test_datasets = datasets.FashionMNIST('./data', train=False, download=False, transform=transform)

train_loader = DataLoader(train_datasets, batch_size=256, shuffle=True)
test_loader = DataLoader(test_datasets, batch_size=256)

In [38]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 6, 5), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Flatten(),
            nn.Linear(16*4*4, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(),
            nn.Linear(84, 10)
        )

    def forward(self, X):
        return self.net(X)

model = LeNet().to(device)

In [39]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [42]:
def train_one_epoch(model, loader):
    model.train()
    total, correct, total_loss = 0, 0, 0

    for X,y in loader:
        X,y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        y_hat = model(X)
        loss = loss_fn(y_hat, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (y_hat.argmax(1) == y).sum().item()
        total += y.size(0)

    return total_loss / len(loader) , correct / total

def evaluate(model, loader):
    model.eval()
    total, correct = 0, 0
    
    with torch.no_grad():
        for X,y in loader:
            X,y = X.to(device), y.to(device)

            pred = model(X)
            correct += (pred.argmax(1) == y).sum().item()
            total += y.size(0)

    return correct / total

In [43]:
for epoch in range(10):
    loss, acc = train_one_epoch(model, train_loader)
    test_acc = evaluate(model, test_loader)
    print(epoch+1, loss, acc, test_acc)

1 0.5450441207023378 0.7914833333333333 0.7945
2 0.47424012473289 0.8239666666666666 0.8391
3 0.4227173419708901 0.8483666666666667 0.8411
4 0.38885841420356265 0.8593666666666666 0.8555
5 0.3623346161969165 0.8697666666666667 0.8633
6 0.34733664437811423 0.8756 0.8611
7 0.3307844322412572 0.88085 0.8617
8 0.31994165188454565 0.8841166666666667 0.8739
9 0.3046964077239341 0.8908166666666667 0.8772
10 0.2974615951167776 0.89255 0.8842
